In [1]:
from pyspark.sql.functions import col

In [2]:
from pyspark.sql.types import StructType, StringType, IntegerType

In [3]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

spark = SparkSession. \
    builder. \
    master('yarn'). \
    config('spark.ui.port', '0'). \
    config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
    config('spark.shuffle.useOldFetchProtocol', 'true'). \
    enableHiveSupport(). \
    getOrCreate()

In [4]:
schema = StructType() \
    .add("order_id", StringType()) \
    .add("product", StringType()) \
    .add("amount", IntegerType())

In [5]:
df = spark.readStream \
    .schema(schema) \
    .json("/user/itv020178/spark_structured")

In [6]:
result_df = df \
    .withWatermark("order_time", "10 minutes") \
    .groupBy(
        window(col("order_time"), "5 minutes"), 
        col("product")
    ) \
    .sum("amount")

In [7]:
query = result_df.writeStream \
    .queryName("my_streaming_table") \
    .outputMode("complete") \
    .format("memory") \
    .trigger(processingTime="10 seconds") \
    .start()

In [8]:
import time

In [9]:
time.sleep(15)

In [11]:
spark.sql("SELECT * FROM my_streaming_table").show()

+-------+-----------+
|product|sum(amount)|
+-------+-----------+
| mobile|      20000|
| laptop|      50000|
+-------+-----------+



In [12]:
spark.sql("SELECT * FROM my_streaming_table").show()

+-------+-----------+
|product|sum(amount)|
+-------+-----------+
| mobile|      35000|
| laptop|     120000|
+-------+-----------+



In [13]:
query.stop()